In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import copy
import gc
import glob
import math
import os
import pathlib
import time
from collections import OrderedDict
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import xarray as xr
from tqdm.notebook import tqdm

plt.rcParams["font.family"] = "serif"
plt.style.use("tableau-colorblind10")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = r":4096:8"  # to make calculations deterministic

In [ ]:
from scripts.train_pdd_lorenz96 import DT, initialize_trainer
from src.configs.lorenz96_config import Lorenz96UnetConfig
from src.models.dynamics.surrogate_simulators import run_simulation
from src.util.random_seed_helper import set_seeds

# Define constants

In [ ]:
DEVICE = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
ROOT_DIR = pathlib.Path(os.environ["PYTHONPATH"].split(":")[0]).resolve()
print(f"{ROOT_DIR=}, {DEVICE=}")

# Plot deep-learning data

In [ ]:
path = f"{ROOT_DIR}/data/DL_data/lorenz96/lorenz96_K32J04_b10p0_c10p0_F10p0_sigma0p0.nc"
da = xr.load_dataarray(path)

In [ ]:
data = da.sel(batch=0).values
assert data.shape == (2, 64, 128)  # channel, time, space

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, ax in enumerate(axes):
    ax.set_xticks([])
    ax.set_yticks([])
    d = data[i].transpose()
    xs = np.arange(d.shape[0])
    ts = np.arange(d.shape[1])
    xs, ts = np.meshgrid(xs, ts, indexing="ij")
    cmap = "BrBG_r" if i == 0 else "twilight_shifted"
    ret = ax.pcolormesh(xs, ts, d, shading="nearest", cmap=cmap)
    fig.colorbar(ret, ax=ax)
plt.show()

# Load a trained PDD

In [ ]:
p = f"{ROOT_DIR}/configs/lorenz96_unet.yml"
config = Lorenz96UnetConfig.load(p)
p = f"{ROOT_DIR}/data/DL_model/lorenz96/lorenz96_unet"
trainer, dataset = initialize_trainer(
    config, str(DEVICE), ROOT_DIR, result_dir=p, kind="test"
)
trainer.load_only_model(milestone=30_000)
_ = trainer.model.noise_estimate_fn.closure.eval()

# Perform simulation

In [ ]:
dict_results = {}
for diffusion_idx in [0, 100, 200]:
    ground_truth, pred = run_simulation(
        trainer=trainer,
        dataset=dataset,
        config=config,
        n_batches=5,
        diffusion_index=diffusion_idx,
        dt=DT,
        n_steps=64,
        device=DEVICE,
        is_noise_off=True,
    )
    dict_results[diffusion_idx] = {"gt": ground_truth, "pred": pred}

In [ ]:
for diffusion_idx in [0, 100, 200]:
    fig, axes = plt.subplots(2, 2, figsize=(10, 4))

    for j, i in product(range(2), range(2)):
        if j == 0:
            data = dict_results[diffusion_idx]["gt"][0][:, :64]
            ttl = "Physics Simulation"
        elif j == 1:
            data = dict_results[diffusion_idx]["pred"][0][:, :64]
            ttl = "Surrogate Simulation"
        assert data.shape == (2, 64, 128)  # channel, time, space
        scaled = dataset.standardize(data)

        ax = axes[j, i]
        ax.set_xticks([])
        ax.set_yticks([])
        d = scaled[i].transpose()

        xs = np.arange(d.shape[0])
        ts = np.arange(d.shape[1])
        xs, ts = np.meshgrid(xs, ts, indexing="ij")
        cmap = "BrBG_r" if i == 0 else "twilight_shifted"
        ret = ax.pcolormesh(xs, ts, d, shading="nearest", cmap=cmap, vmin=-2, vmax=2)
        fig.colorbar(ret, ax=ax)
        ax.set_title(ttl)

    plt.suptitle(f"diffusion scale = {float(diffusion_idx)/config.num_timesteps:.2f}")
    plt.show()

# Perform generation

In [ ]:
set_seeds(42)
intermediates = trainer.model.sample(
    batch_size=5,
    corrector_snr=0.7,
    num_corrector_steps=3,
)

In [ ]:
data = intermediates[1][4].numpy()
assert data.shape == (2, 64, 128)  # channel, time, space

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, ax in enumerate(axes):
    ax.set_xticks([])
    ax.set_yticks([])
    d = data[i].transpose()
    xs = np.arange(d.shape[0])
    ts = np.arange(d.shape[1])
    xs, ts = np.meshgrid(xs, ts, indexing="ij")
    cmap = "BrBG_r" if i == 0 else "twilight_shifted"
    ret = ax.pcolormesh(xs, ts, d, shading="nearest", cmap=cmap, vmin=-2, vmax=2)
    fig.colorbar(ret, ax=ax)
plt.show()

# Perform super-resolution

In [ ]:
set_seeds(42)
_, lr = run_simulation(
    trainer=trainer,
    dataset=dataset,
    config=config,
    n_batches=5,
    diffusion_index=200,
    dt=DT,
    n_steps=128,
    device=DEVICE,
    is_noise_off=False,
)
lr = dataset.standardize(lr[:, :, -64:, :])

In [ ]:
set_seeds(42)
intermediates = trainer.model._p_loop(
    n_timesteps=201,
    n_batches=5,
    img=torch.from_numpy(lr).to(device=DEVICE, dtype=torch.float32),
    num_corrector_steps=3,
    corrector_snr=0.7,
)

In [ ]:
sr = intermediates[1].numpy()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 4))

for j, i in product(range(2), range(2)):
    if j == 0:
        scaled = sr[0]
        ttl = "Super-Resolution"
    elif j == 1:
        scaled = lr[0]
        ttl = "Low-Resolution"
    assert data.shape == (2, 64, 128)  # channel, time, space

    ax = axes[j, i]
    ax.set_xticks([])
    ax.set_yticks([])
    d = scaled[i].transpose()

    xs = np.arange(d.shape[0])
    ts = np.arange(d.shape[1])
    xs, ts = np.meshgrid(xs, ts, indexing="ij")
    cmap = "BrBG_r" if i == 0 else "twilight_shifted"
    ret = ax.pcolormesh(xs, ts, d, shading="nearest", cmap=cmap, vmin=-2, vmax=2)
    fig.colorbar(ret, ax=ax)
    ax.set_title(ttl)

plt.show()